# 대출 적합성 심사 에이전트 — CrewAI 3-Agent
# 작성자 : 원광식 
> **LLM · Transformer 기반 3-Agent 파이프라인 · 교육용 데모**
> 고객 자연어 입력 → 3개 Agent 협업 → 승인가능 / 상담필요 / 어려움 판정 + 맞춤 안내문

> ⚠️ **본 서비스는 교육용 실습 데모입니다.** 더미 데이터·가상 고객만 사용하며,
> 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.

---

### 이 노트북 실행 순서 (텍스트 탭 = 각 코드 셀 위의 설명)
1. **설치 · 환경** — 패키지 설치, `.env` 안내
2. **[API 키 불필요] 데이터 · 결정적 심사 로직 · 자체 테스트** — 셀 4~7 (키 없이 3/3 확인 가능)
3. **[API 키 필요] 3-Agent 파이프라인** — 셀 8~ (환경변수 로드 → 도구 → Agent → Task → Crew → 실행)

> 💡 셀을 **위에서부터 순서대로** 실행하세요. 2번 블록은 API 키 없이도 판정 로직을 검증합니다.

## 1. 패키지 설치

`code_check`의 예시(`Input-CSV-Fashion_Coordinator.ipynb`)와 동일하게 CrewAI·dotenv를 설치합니다.
설치 후 커널을 한 번 재시작하는 것이 안전합니다. (pandas 없이도 동작하도록 CSV는 표준 라이브러리로 파싱)

In [1]:
# 노트북 커널에 CrewAI, 도구 패키지, 환경변수 로더를 설치합니다.
# 설치 후에는 Jupyter 커널을 한 번 재시작하는 것이 안전합니다.
%pip install -U "crewai[openai,tools]>=1.15,<2.0" "python-dotenv>=1.0"

  Using cached pydantic_settings-2.14.2-py3-none-any.whl.metadata (3.4 kB)


  Using cached beautifulsoup4-4.13.5-py3-none-any.whl.metadata (3.8 kB)
  Using cached pymupdf-1.26.7-cp310-abi3-macosx_11_0_arm64.whl.metadata (3.4 kB)


  Using cached python_docx-1.2.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached pytube-15.0.0-py3-none-any.whl.metadata (5.0 kB)
  Using cached tiktoken-0.12.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached youtube_transcript_api-1.2.4-py3-none-any.whl.metadata (24 kB)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/811.5 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.5/811.5 kB 49.2 MB/s  0:00:00
Using cached beautifulsoup4-4.13.5-py3-none-any.whl (105 kB)
Using cached pydantic_settings-2.14.2-py3-none-any.whl (61 kB)
Using cached pymupdf-1.26.7-cp310-abi3-macosx_11_0_arm64.whl (22.5 MB)
Using cached python_docx-1.2.0-py3-none-any.whl (252 kB)
Using cached pytube-15.0.0-py3-none-any.whl (57 kB)
Using cached tiktoken-0.12.0-cp311-cp311-macosx_11_0_arm64.whl (995 kB)
Using cached youtube_transcript_api-1.2.4-py3-none-any.whl (485 kB)


...

      Successfully uninstalled crewai-1.15.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 10/12 [crewai]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 10/12 [crewai]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 10/12 [crewai]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12/12 [crewai-tools]


Note: you may need to restart the kernel to use updated packages.


## 2. `.env` 파일 예시

이 노트북과 **같은 폴더**에 `.env` 파일을 만들고 아래 값을 입력합니다. (`.env.example` 참고)

```dotenv
OPENAI_API_KEY=sk-proj-실제_API_KEY
OPENAI_MODEL_NAME=openai/gpt-4o-mini
```

> 🔐 API 키는 **절대 코드/노트북에 직접 쓰지 않습니다.** `.env`에만 두고, 커밋·공유 금지.
> (참고 예시 노트북들은 키를 하드코딩했지만, 본 서비스는 규제 통제상 `.env` 방식만 사용합니다.)

## 3. 공통 설정 · 더미 대출상품 로드  *(API 키 불필요)*

- `DISCLAIMER`: 모든 안내문에 강제 삽입되는 교육용 디스클레이머(규제 통제 R-3).
- `load_products()`: `loan_products.csv`(진실의 원천)를 파싱. 금리 `4.5%`→`4.5`, `3등급이상`→`3`(숫자 낮을수록 우량)으로 정규화.

> **텍스트 탭 — 차이점:** 참고 예시는 pandas로 CSV를 읽지만, 여기서는 의존성을 줄이려 표준 라이브러리로 파싱합니다.

In [2]:
import json
from loan_agent.core import BASE_DIR, CSV_PATH, DISCLAIMER, PRODUCTS

for p in PRODUCTS:
    print(p)

{'상품코드': 'A-01', '상품명': '무담보신용대출', '은행': 'A은행', '최저금리': 4.5, '최고금리': 8.0, '최대한도': 50000000, '필요신용등급': 3, '담보필요': False, '직장조건': '정규직'}
{'상품코드': 'A-02', '상품명': '프리미엄대출', '은행': 'A은행', '최저금리': 3.0, '최고금리': 5.5, '최대한도': 100000000, '필요신용등급': 1, '담보필요': False, '직장조건': '정규직'}
{'상품코드': 'A-03', '상품명': '담보론', '은행': 'A은행', '최저금리': 3.8, '최고금리': 6.5, '최대한도': 250000000, '필요신용등급': 4, '담보필요': True, '직장조건': '제한없음'}
{'상품코드': 'A-04', '상품명': '소액대출', '은행': 'A은행', '최저금리': 10.0, '최고금리': 16.0, '최대한도': 7000000, '필요신용등급': 6, '담보필요': False, '직장조건': '제한없음'}
{'상품코드': 'B-01', '상품명': '무담보신용대출', '은행': 'B은행', '최저금리': 6.0, '최고금리': 12.0, '최대한도': 30000000, '필요신용등급': 5, '담보필요': False, '직장조건': '제한없음'}
{'상품코드': 'B-02', '상품명': '우량고객대출', '은행': 'B은행', '최저금리': 3.5, '최고금리': 6.0, '최대한도': 80000000, '필요신용등급': 2, '담보필요': False, '직장조건': '정규직'}
{'상품코드': 'B-03', '상품명': '담보부대출', '은행': 'B은행', '최저금리': 4.0, '최고금리': 7.0, '최대한도': 300000000, '필요신용등급': 3, '담보필요': True, '직장조건': '제한없음'}
{'상품코드': 'B-04', '상품명': '급여소액대출', '은행': 'B은행', '최저금리': 9.0,

## 4. 결정적 적합성 심사 로직  *(API 키 불필요 · 핵심 안전장치)*

> **텍스트 탭 — 이 서비스의 가장 큰 설계 차이점:**
> 판정을 **LLM이 아니라 이 결정적 함수(`screen_loan`)가 내립니다.** CSV 하드규칙(신용등급·담보·직장조건·한도)과
> DSR(연간 원리금상환액÷연소득)로 판정하므로, 단일 모델 SC의 오류 상관(R-6)이나 환각(R-2)이 최종 판정을 흔들지 못합니다.
> LLM 3-Agent는 이 판정의 **근거 설명·안내문 작성**만 담당합니다.

**판정 규칙**
- 적격 상품 = 신용등급·담보·직장조건·희망금액(≤한도) 모두 충족
- 상환능력(DSR): ≤0.30 여유 / ≤0.40 보통 / >0.40 부족
- 최종: 적격 없음 or 부족 → **어려움** / 여유 → **승인가능** / 보통 → **상담필요**

> **[디벨롭] DSR 고도화 노트:** 초기 버전은 상환능력을 간이 DTI(`기존부채 ÷ 연소득`)로만 봐서, 이번에 신청하는
> 희망금액의 상환부담을 반영하지 못했습니다(기존부채가 0이면 금액과 무관하게 항상 "여유"). 이를 금융위 실제 정의인
> **DSR = 연간 원리금상환액 ÷ 연소득**으로 교체했습니다. 기존부채와 신규 희망금액을 모두 **원리금균등상환**으로
> 월상환액을 산출해 합산하며, 판정 밴드 0.40은 **은행권 실제 DSR 규제 상한**을 반영한 값입니다.
> 남은 단순화: 대표 고정금리(연 6%)·고정기간(60개월)을 가정하며 개별 상품의 실제 금리는 반영하지 않습니다.

In [3]:
from loan_agent.core import screen_loan

# 간단 확인
print(json.dumps(screen_loan({"월소득": 7000000, "부채": 0, "신용등급": 1,
                              "희망금액": 30000000, "직장유형": "정규직"}),
                 ensure_ascii=False, indent=2))

{
  "판정": "승인가능",
  "상환능력": "여유",
  "DSR": 0.083,
  "월상환액": {
    "기존부채": 0,
    "신규대출": 579984,
    "합계": 579984,
    "가정": {
      "연금리": 0.06,
      "기간개월": 60
    }
  },
  "적격상품": [
    {
      "상품코드": "A-01",
      "상품명": "무담보신용대출",
      "은행": "A은행"
    },
    {
      "상품코드": "A-02",
      "상품명": "프리미엄대출",
      "은행": "A은행"
    },
    {
      "상품코드": "B-01",
      "상품명": "무담보신용대출",
      "은행": "B은행"
    },
    {
      "상품코드": "B-02",
      "상품명": "우량고객대출",
      "은행": "B은행"
    },
    {
      "상품코드": "C-02",
      "상품명": "신용대출",
      "은행": "C은행"
    },
    {
      "상품코드": "D-02",
      "상품명": "정규직우대대출",
      "은행": "D은행"
    },
    {
      "상품코드": "E-01",
      "상품명": "직장인우대대출",
      "은행": "E은행"
    },
    {
      "상품코드": "F-03",
      "상품명": "무담보신용대출",
      "은행": "F은행"
    }
  ],
  "부적격사유": {
    "A-03 담보론(A은행)": [
      "담보 필요(미보유)"
    ],
    "A-04 소액대출(A은행)": [
      "희망금액 초과(한도 7,000,000원)"
    ],
    "B-03 담보부대출(B은행)": [
      "담보 필요(미보유)"
    ],
    "B-04 급여소액대출(B은행)": 

## 5. 자연어 → 구조화 파싱 (규칙 기반 폴백)  *(API 키 불필요)*

> **텍스트 탭 — 차이점:** 실제 파싱은 아래 **Agent 1(LLM)** 이 담당합니다. 이 규칙 기반 파서는
> ① API 키 없이 로직을 검증하고 ② LLM 파싱 결과를 교차 확인하기 위한 **폴백**입니다.

In [4]:
from loan_agent.core import parse_korean_amount, rule_based_parse

print(rule_based_parse("월급 700만원 받는 정규직이고 부채는 없습니다. 신용등급 1등급이고 3000만원 대출받고 싶어요."))

{'월소득': 7000000, '부채': 0, '신용등급': 1, '희망금액': 30000000, '직장유형': '정규직', '담보보유': False}


## 6. 테스트 케이스 3종 · 로직 자체 테스트  *(API 키 불필요 — 여기서 3/3 확인)*

기획서의 데모 3종입니다. **여기까지는 API 키 없이 실행**되며, 결정적 판정 로직만으로 3/3 일치를 확인합니다.

In [5]:
from loan_agent.core import TEST_CASES, run_logic_selftest

run_logic_selftest()

결정적 심사 로직 자체 테스트 (API 키 불필요)

[승인 케이스] ✅
  파싱 : {'월소득': 7000000, '부채': 0, '신용등급': 1, '희망금액': 30000000, '직장유형': '정규직', '담보보유': False}
  기대 : 승인가능 / 판정: 승인가능 (상환 여유, DSR 0.083)
  적격 : [{'상품코드': 'A-01', '상품명': '무담보신용대출', '은행': 'A은행'}, {'상품코드': 'A-02', '상품명': '프리미엄대출', '은행': 'A은행'}, {'상품코드': 'B-01', '상품명': '무담보신용대출', '은행': 'B은행'}, {'상품코드': 'B-02', '상품명': '우량고객대출', '은행': 'B은행'}, {'상품코드': 'C-02', '상품명': '신용대출', '은행': 'C은행'}, {'상품코드': 'D-02', '상품명': '정규직우대대출', '은행': 'D은행'}, {'상품코드': 'E-01', '상품명': '직장인우대대출', '은행': 'E은행'}, {'상품코드': 'F-03', '상품명': '무담보신용대출', '은행': 'F은행'}]

[상담필요 케이스] ✅
  파싱 : {'월소득': 2500000, '부채': 20000000, '신용등급': 4, '희망금액': 25000000, '직장유형': '제한없음', '담보보유': False}
  기대 : 상담필요 / 판정: 상담필요 (상환 보통, DSR 0.348)
  적격 : [{'상품코드': 'B-01', '상품명': '무담보신용대출', '은행': 'B은행'}, {'상품코드': 'E-04', '상품명': '중금리대출', '은행': 'E은행'}]

[어려움 케이스] ✅
  파싱 : {'월소득': 1800000, '부채': 30000000, '신용등급': 6, '희망금액': 10000000, '직장유형': '제한없음', '담보보유': False}
  기대 : 어려움 / 판정: 어려움 (상환 부족, DSR 0.43)
  적격 : [{'상품코드': '

True

### 6-1. 엣지케이스 2종 · 로직 자체 테스트  *(API 키 불필요)*

`EDGE_CASES`(담보 보유 저신용 / 계약직 소액)도 동일한 결정적 로직으로 검증합니다. 정의만 해두고 실행하지 않으면 설계 의도(담보·직장조건 하드규칙)가 검증되지 않으므로, 여기서 2/2 일치를 직접 확인합니다.

In [6]:
from loan_agent.core import EDGE_CASES

run_logic_selftest(EDGE_CASES)

결정적 심사 로직 자체 테스트 (API 키 불필요)

[담보 보유 저신용 케이스] ✅
  파싱 : {'월소득': 3000000, '부채': 5000000, '신용등급': 8, '희망금액': 8000000, '직장유형': '제한없음', '담보보유': True}
  기대 : 승인가능 / 판정: 승인가능 (상환 여유, DSR 0.084)
  적격 : [{'상품코드': 'D-04', '상품명': '초저신용특별대출', '은행': 'D은행'}]

[계약직 소액 케이스] ✅
  파싱 : {'월소득': 1500000, '부채': 2000000, '신용등급': 6, '희망금액': 5000000, '직장유형': '계약직', '담보보유': False}
  기대 : 승인가능 / 판정: 승인가능 (상환 여유, DSR 0.09)
  적격 : [{'상품코드': 'A-04', '상품명': '소액대출', '은행': 'A은행'}, {'상품코드': 'D-01', '상품명': '중금리대출', '은행': 'D은행'}, {'상품코드': 'E-02', '상품명': '무직자소액대출', '은행': 'E은행'}, {'상품코드': 'E-04', '상품명': '중금리대출', '은행': 'E은행'}, {'상품코드': 'F-02', '상품명': '계약직소액대출', '은행': 'F은행'}]

결과: 전체 통과 (2/2)


True

### 6-2. 필수 입력 검증 · 자체 테스트  *(API 키 불필요)*

> **타팀 피드백 반영:** 사용자가 필수 정보(월 소득·신용등급·희망 대출금액)를 입력하지 않으면, 예전엔 Agent가 기본값(sentinel)으로 채워 `screen_loan`이 그대로 **'어려움'** 을 내면서 *정보 부족*을 *거절*로 오판했습니다.
>
> `core.missing_required_fields()`가 심사 **전에** 누락 필드를 잡아냅니다. Streamlit 앱은 이 결과로 심사를 막고 재입력을 안내합니다. 여기서 정상 5종은 통과(누락 없음), 누락 4종은 정확히 감지되는지 확인합니다.

In [7]:
from loan_agent.core import missing_required_fields, rule_based_parse

print('=' * 60)
print('필수 입력 검증 자체 테스트 (API 키 불필요)')
print('=' * 60)

print('\n[1] 정상 케이스 — 누락이 없어야 함(심사 진행 가능):')
for tc in TEST_CASES + EDGE_CASES:
    miss = missing_required_fields(rule_based_parse(tc['input']))
    print(f"  {'✅' if not miss else '❌'} {tc['name']:16s} 누락필드: {miss or '없음'}")

print('\n[2] 필수정보 누락 케이스 — 빠진 항목이 정확히 감지되어야 함:')
missing_inputs = [
    ('신용등급·희망금액 누락', '월급 300만원 받는 정규직이고 부채는 없습니다.'),
    ('소득 누락',           '신용등급 3등급이고 2000만원 대출받고 싶어요.'),
    ('희망금액 누락',        '월급 400만원, 신용등급 2등급입니다.'),
    ('상담 요청만 함',       '안녕하세요, 대출 상담 받고 싶어요.'),
]
for name, text in missing_inputs:
    miss = missing_required_fields(rule_based_parse(text))
    print(f"  {'✅' if miss else '❌'} {name:16s} 누락필드: {miss}")

print('\n' + '=' * 60)
print('→ 정상 입력은 통과, 누락 입력은 어떤 필드가 빠졌는지 안내 (앱에서 재입력 유도)')
print('=' * 60)

필수 입력 검증 자체 테스트 (API 키 불필요)

[1] 정상 케이스 — 누락이 없어야 함(심사 진행 가능):
  ✅ 승인 케이스           누락필드: 없음
  ✅ 상담필요 케이스         누락필드: 없음
  ✅ 어려움 케이스          누락필드: 없음
  ✅ 담보 보유 저신용 케이스    누락필드: 없음
  ✅ 계약직 소액 케이스       누락필드: 없음

[2] 필수정보 누락 케이스 — 빠진 항목이 정확히 감지되어야 함:
  ✅ 신용등급·희망금액 누락     누락필드: ['신용등급', '희망 대출금액']
  ✅ 소득 누락            누락필드: ['월 소득']
  ✅ 희망금액 누락          누락필드: ['희망 대출금액']
  ✅ 상담 요청만 함         누락필드: ['월 소득', '신용등급', '희망 대출금액']

→ 정상 입력은 통과, 누락 입력은 어떤 필드가 빠졌는지 안내 (앱에서 재입력 유도)


---
# ⬇️ 여기부터 API 키 필요 — 3-Agent 파이프라인

아래 셀부터는 `.env`의 `OPENAI_API_KEY`가 필요합니다. 키가 없으면 다음 셀에서 명확한 오류가 발생합니다.

## 7. 환경변수 로드 · 공통 LLM 객체

> **텍스트 탭 — 차이점:** 참고 예시(`Fashion_Coordinator`)와 동일한 패턴이지만 모델을 **GPT-4o-mini**로 고정하고,
> 키가 없으면 즉시 `ValueError`로 중단해 원인을 명확히 알립니다. 이 `llm` 객체를 3개 Agent가 공유합니다.

In [8]:
from loan_agent.core import get_llm

llm, model_name = get_llm()
print(f"사용 모델: {model_name}")

사용 모델: openai/gpt-4o-mini


## 8. 도구(Tool) 정의 — ReAct 조회 · 하드규칙 심사

> **텍스트 탭 — 차이점(중요):** CrewAI가 OpenAI에 함수를 넘길 때 **툴 이름은 영숫자여야** 합니다.
> 한글 툴명은 빈 문자열로 처리되어 `OpenAI function name cannot be empty` 오류가 납니다.
> 그래서 툴 이름은 ASCII(`lookup_loan_product`, `assess_loan_eligibility`)로 두고, **설명·인자는 한글**로 둡니다.
>
> - `lookup_loan_product` : Agent 3의 **ReAct** 조회 도구(금리·한도 재확인, CSV 외 값 반환 금지).
> - `assess_loan_eligibility` : Agent 2가 호출하는 **결정적 판정** 도구(4장의 `screen_loan`을 감쌈 → LLM보다 우선).

In [9]:
from loan_agent.core import lookup_product, screen_tool

print("도구 2종 정의 완료:", [lookup_product.name, screen_tool.name])

도구 2종 정의 완료: ['lookup_loan_product', 'assess_loan_eligibility']


## 9. Agent 1 — 정보 파싱가

고객 자연어에서 5필드를 추출해 **원 단위 JSON**으로 구조화합니다. 입력에 없는 값은 지어내지 않습니다.

> **수업 개념(Self-Attention):** LLM 호출 시 프롬프트 전체 토큰이 한 번에 서로를 참조 → "월급·부채·희망금액" 관계를 동시 포착.

In [10]:
from loan_agent.core import build_parser_agent

parser = build_parser_agent(llm)

## 10. Agent 2 — 심사 판단가 (CoT + Self-Consistency)

> **텍스트 탭 — 차이점:** 이 Agent는 **먼저 `assess_loan_eligibility` 도구로 결정적 판정을 받고 그것을 최종 결론으로** 삼습니다.
> 그 위에서 **CoT 4단계**(①부채비율 ②신용등급 ③한도 ④상품선별)로 근거를 설명하고, **SC 3관점**(보수·낙관·중립)으로 교차검증합니다.
> → 단일 모델 SC의 오류 상관 위험을 CSV 하드규칙이 차단합니다.
>
> **개념 정확도 노트:** 엄밀한 Self-Consistency는 "동일 질문을 여러 번 독립적으로 추론시킨 뒤 다수결로 답을 고르는" 기법입니다(여러 번의 별도 LLM 호출 필요).
> 이 노트북은 **1회 호출 안에서 세 관점(보수·낙관·중립)을 서술하게 하는 다각도 프롬프팅**으로 SC의 아이디어를 응용했습니다 — 최종 판정 자체는 SC가 아니라
> 결정적 로직(`screen_loan`)이 확정하므로, 이 응용이 판정의 안전성에 영향을 주지 않습니다. SC를 "판정 방법"이 아니라 "근거 설명의 다각도 검증"에 쓴 것이 이 설계의 의도입니다.

In [11]:
from loan_agent.core import build_reviewer_agent

reviewer = build_reviewer_agent(llm)

## 11. Agent 3 — 결과 안내가 (ReAct)

> **텍스트 탭 — 차이점:** 금리·한도를 안내문에 쓰기 전에 **반드시 `lookup_loan_product`로 재확인(ReAct)** 하고,
> **확정 표현 금지**(→ "~로 판단됩니다(데모 기준)"), **마지막 줄에 디스클레이머 강제**(규제 통제 R-3).

In [12]:
from loan_agent.core import build_advisor_agent

advisor = build_advisor_agent(llm)

## 12. Task 정의 — 파싱 → 심사 → 안내

`context`로 앞 Task의 출력을 다음 Task 입력으로 넘겨 **순차 연동(R4)** 합니다.

In [13]:
from loan_agent.core import build_tasks

parse_task, review_task, advise_task = build_tasks(parser, reviewer, advisor)
print("Task 3종 정의 완료")

Task 3종 정의 완료


## 13. Crew 구성 — 순차 실행

Agent 1 → 2 → 3 을 `Process.sequential` 로 연결합니다.

In [14]:
from crewai import Crew, Process

crew = Crew(
    agents=[parser, reviewer, advisor],
    tasks=[parse_task, review_task, advise_task],
    process=Process.sequential,
    verbose=True,
)
print("Crew 구성 완료")

Crew 구성 완료


## 14. 파이프라인 실행 · 토큰 Usage 관측

> **수업 개념(추론 파이프라인 비용):** 3-Agent 순차 호출마다 토큰화→임베딩→Self-Attention 반복→자기회귀 생성이
> 처음부터 수행됩니다. `usage_metrics`로 **토큰 사용량(비용)** 을 직접 관측해 다단계 호출의 트레이드오프를 체감합니다(R7).
>
> **참고:** 이 토큰 Usage 관측은 이 노트북(개발·수업용 산출물)에서 확인합니다. `loan_agent/app.py`(Streamlit 데모)는
> 실제 고객이 보는 화면을 가정해 UX상 토큰 수치를 노출하지 않도록 설계했습니다 — 같은 `core.py`의 `usage_metrics`를
> 노트북은 그대로 보여주고, 웹앱은 의도적으로 숨기는 것으로 역할을 나눴습니다.

In [15]:
from IPython.display import Markdown, display

async def run_service(customer_input: str):
    """전체 3-Agent 파이프라인을 1회 실행하고 결과·토큰 Usage를 출력."""
    result = await crew.kickoff_async(inputs={"customer_input": customer_input})
    usage = getattr(crew, "usage_metrics", None)
    tasks_output = getattr(result, "tasks_output", None) or []
    print("\n" + "-" * 60)
    print("토큰 Usage (추론 파이프라인 비용 체감 — Self-Attention/추론 반복):")
    print(f"  {usage}")
    print("-" * 60)
    return {
        "파싱결과": tasks_output[0].raw if len(tasks_output) > 0 else None,
        "심사결과": tasks_output[1].raw if len(tasks_output) > 1 else None,
        "안내문": str(result),
        "usage": usage,
    }

### 14-1. 단일 케이스 실행 (승인 케이스)

먼저 한 케이스만 실행해 파이프라인 동작과 최종 안내문을 확인합니다.

In [16]:
out = await run_service(TEST_CASES[0]["input"])
display(Markdown("### 최종 안내문\n\n" + out["안내문"]))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 94613b06-2629-4458-a7b7-1bab256ddde0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 고객 입력을 파싱하라:                                                                               │
│  "월급 700만원 받는 정규직이고 부채는 없습니다. 신용등급 1등급이고 3000만원 대출받고 싶어요."                   │
│                                                                                                                 │
│  월소득·부채·신용등급·희망금액·직장유형·담보보유 6개 필드를 추출하라. 금액은 원 단위 정수로                     │
│  환산하라(700만원->7000000). 부채가 '없다'면 0. 직장유형은 정규직/계약직/제한없음 중 하나. 담보보유는 담보      │
│  제공 의사가 명시된 경우에만 true, 그 외 false.                                                                 │
│  ID: f0c7dd0f-a1f6-4f75-bc4c-b2d5a2fb417d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 정보 파싱가                                                                                             │
│                                                                                                                 │
│  Task: 다음 고객 입력을 파싱하라:                                                                               │
│  "월급 700만원 받는 정규직이고 부채는 없습니다. 신용등급 1등급이고 3000만원 대출받고 싶어요."                   │
│                                                                                                                 │
│  월소득·부채·신용등급·희망금액·직장유형·담보보유 6개 필드를 추출하라. 금액은 원 단위 정수로                     │
│  환산하라(700만원->7000000). 부채가 '없다'면 0. 직장유형은 정규직/계약직/제한없음 중 하나. 담보보유는 담보      │
│  제공 의사가 명시된 경우에만 true, 그 외 false.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 정보 파싱가                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  {"월소득":7000000,"부채":0,"신용등급":1,"희망금액":30000000,"직장유형":"정규직","담보보유":false}              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 결과 안내가                                                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  검토 결과 승인 가능한 것으로 판단됩니다(데모 기준).                                                            │
│                                                                                                                 │
│  추천 상품은 다음과 같습니다:                                                                                   │
│  - **상품명**: 프리미엄대출                                                                                     │
│  - **상품코드**: A-02                                                                                           │
│  - **은행**: A은행                                                                                              │
│  - **금리**: 3.0% ~ 5.5%                                                                                        │
│  - **최대한도**: 100,000,000원                                                                                  │
│                                                                                                                 │
│  이 상품은 고객님의 신용등급과 소득에 적합하며, 대출 한도와 금리 조건도 유리합니다.                             │
│                                                                                                                 │
│  본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Agent 2의 판정과 근거를 받아 고객용 안내문을 작성하라. Agent 2가 전달한 '추천상품'(도구가 최저금리       │
│  기준으로 확정한 시스템 추천)을 최우선으로 그대로 안내하라 — 스스로 다른 상품을 계산해 대체하거나 추가로 끼워   │
│  넣지 마라. 그 추천 상품 하나만 'lookup_loan_product' 도구로 조회해 금리·한도를 재확인하라(ReAct) — 불필요한    │
│  반복 조회로 토큰과 응답 시간을 낭비하지 않는다. 판정 라벨에 맞춰 톤과 첫 문장을 다르게 하라(라벨마다 하나만):  │
│  '승인가능'이면 긍정적 톤으로 시작하고 '상담이 필요하다'는 문장은 넣지 마라. '상담필요'이면 보완하면 승인       │
│  가능성이 있는 중립적 상태이니 '승인 가능성이 낮은 상황입니다' 같은 부정적 문장 대신 '추가로 확인·보완이        │
│  필요한 부분이 있어 상담을 안내드립니다'처럼 중립적으로 시작하라. '어려움'이면 '추천상품'이 없으므로(None)      │
│  상품을 나열하거나 '추천'하지 말고, '현재 기준으로는 승인이 어려운 것으로 판단됩니다(데모 기준)'처럼 시작한 뒤  │
│  상환능력·신용등급 개선 방향(부채 축소, 소득 안정화, 담보 제공, 소액부터 재신청 등)과 상담 채널 안내로          │
│  마무리하라. 확인된 수치만 사용하고, 확정 표현 대신 조건부 표현을 쓰라. 같은 이름의 상품이 여러 은행에 있을 수  │
│  있으니, 상품을 언급할 때 상품코드와 은행명을 함께 명시하라. 안내문 마지막 줄에 다음을 그대로 포함하라: "본     │
│  안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다."     │
│  Agent: 결과 안내가                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


------------------------------------------------------------
토큰 Usage (추론 파이프라인 비용 체감 — Self-Attention/추론 반복):
  total_tokens=29550 prompt_tokens=25293 cached_prompt_tokens=10368 completion_tokens=4257 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=15
------------------------------------------------------------


### 최종 안내문

검토 결과 승인 가능한 것으로 판단됩니다(데모 기준). 

추천 상품은 다음과 같습니다:
- **상품명**: 프리미엄대출
- **상품코드**: A-02
- **은행**: A은행
- **금리**: 3.0% ~ 5.5%
- **최대한도**: 100,000,000원

이 상품은 고객님의 신용등급과 소득에 적합하며, 대출 한도와 금리 조건도 유리합니다. 

본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 14-2. 테스트 3종 전체 실행

승인 / 상담필요 / 어려움 3케이스를 모두 실행해 판정과 안내문을 확인합니다. (LLM 호출이 많아 시간이 걸립니다)

In [17]:
for tc in TEST_CASES:
    print(f"\n\n########## {tc['name']} (기대: {tc['expected']}) ##########")
    out = await run_service(tc["input"])
    display(Markdown(f"#### [{tc['name']}] 최종 안내문\n\n" + out["안내문"]))



########## 승인 케이스 (기대: 승인가능) ##########


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 94613b06-2629-4458-a7b7-1bab256ddde0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 고객 입력을 파싱하라:                                                                               │
│  "월급 700만원 받는 정규직이고 부채는 없습니다. 신용등급 1등급이고 3000만원 대출받고 싶어요."                   │
│                                                                                                                 │
│  월소득·부채·신용등급·희망금액·직장유형·담보보유 6개 필드를 추출하라. 금액은 원 단위 정수로                     │
│  환산하라(700만원->7000000). 부채가 '없다'면 0. 직장유형은 정규직/계약직/제한없음 중 하나. 담보보유는 담보      │
│  제공 의사가 명시된 경우에만 true, 그 외 false.                                                                 │
│  ID: f0c7dd0f-a1f6-4f75-bc4c-b2d5a2fb417d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 정보 파싱가                                                                                             │
│                                                                                                                 │
│  Task: 다음 고객 입력을 파싱하라:                                                                               │
│  "월급 700만원 받는 정규직이고 부채는 없습니다. 신용등급 1등급이고 3000만원 대출받고 싶어요."                   │
│                                                                                                                 │
│  월소득·부채·신용등급·희망금액·직장유형·담보보유 6개 필드를 추출하라. 금액은 원 단위 정수로                     │
│  환산하라(700만원->7000000). 부채가 '없다'면 0. 직장유형은 정규직/계약직/제한없음 중 하나. 담보보유는 담보      │
│  제공 의사가 명시된 경우에만 true, 그 외 false.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Agent 2의 판정과 근거를 받아 고객용 안내문을 작성하라. Agent 2가 전달한 '추천상품'(도구가 최저금리       │
│  기준으로 확정한 시스템 추천)을 최우선으로 그대로 안내하라 — 스스로 다른 상품을 계산해 대체하거나 추가로 끼워   │
│  넣지 마라. 그 추천 상품 하나만 'lookup_loan_product' 도구로 조회해 금리·한도를 재확인하라(ReAct) — 불필요한    │
│  반복 조회로 토큰과 응답 시간을 낭비하지 않는다. 판정 라벨에 맞춰 톤과 첫 문장을 다르게 하라(라벨마다 하나만):  │
│  '승인가능'이면 긍정적 톤으로 시작하고 '상담이 필요하다'는 문장은 넣지 마라. '상담필요'이면 보완하면 승인       │
│  가능성이 있는 중립적 상태이니 '승인 가능성이 낮은 상황입니다' 같은 부정적 문장 대신 '추가로 확인·보완이        │
│  필요한 부분이 있어 상담을 안내드립니다'처럼 중립적으로 시작하라. '어려움'이면 '추천상품'이 없으므로(None)      │
│  상품을 나열하거나 '추천'하지 말고, '현재 기준으로는 승인이 어려운 것으로 판단됩니다(데모 기준)'처럼 시작한 뒤  │
│  상환능력·신용등급 개선 방향(부채 축소, 소득 안정화, 담보 제공, 소액부터 재신청 등)과 상담 채널 안내로          │
│  마무리하라. 확인된 수치만 사용하고, 확정 표현 대신 조건부 표현을 쓰라. 같은 이름의 상품이 여러 은행에 있을 수  │
│  있으니, 상품을 언급할 때 상품코드와 은행명을 함께 명시하라. 안내문 마지막 줄에 다음을 그대로 포함하라: "본     │
│  안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다."     │
│  Agent: 결과 안내가                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


------------------------------------------------------------
토큰 Usage (추론 파이프라인 비용 체감 — Self-Attention/추론 반복):
  total_tokens=124614 prompt_tokens=105978 cached_prompt_tokens=44928 completion_tokens=18636 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=60
------------------------------------------------------------


#### [어려움 케이스] 최종 안내문

현재 기준으로는 승인이 어려운 것으로 판단됩니다(데모 기준). 고객님의 DSR(부채상환비율)이 43%로, 일반적으로 허용되는 기준을 초과하여 상환능력이 부족합니다. 또한, 신용등급이 6등급으로 많은 대출 상품에서 요구하는 최소 기준에 미치지 못하고 있습니다. 희망하시는 대출 금액도 적격 상품의 한도를 초과하고 있어 대출 승인 가능성이 낮습니다.

상환능력과 신용등급을 개선하기 위해 부채를 축소하거나 소득을 안정화하는 방법을 고려해 보시기 바랍니다. 담보를 제공하거나 소액부터 재신청하는 것도 좋은 방법입니다. 추가적인 상담이 필요하시면 언제든지 문의해 주시기 바랍니다.

본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 94613b06-2629-4458-a7b7-1bab256ddde0                                                                       │
│  Final Output: 현재 기준으로는 승인이 어려운 것으로 판단됩니다(데모 기준). 고객님의 DSR(부채상환비율)이 43%로,  │
│  일반적으로 허용되는 기준을 초과하여 상환능력이 부족합니다. 또한, 신용등급이 6등급으로 많은 대출 상품에서       │
│  요구하는 최소 기준에 미치지 못하고 있습니다. 희망하시는 대출 금액도 적격 상품의 한도를 초과하고 있어 대출      │
│  승인 가능성이 낮습니다.                                                                                        │
│                                                                                                                 │
│  상환능력과 신용등급을 개선하기 위해 부채를 축소하거나 소득을 안정화하는 방법을 고려해 보시기 바랍니다. 담보를  │
│  제공하거나 소액부터 재신청하는 것도 좋은 방법입니다. 추가적인 상담이 필요하시면 언제든지 문의해 주시기         │
│  바랍니다.                                                                                                      │
│                                                                                                                 │
│  본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 14-3. 엣지케이스 2종 실행 (3-Agent 파이프라인)

담보 보유 저신용 / 계약직 소액 엣지케이스도 3-Agent 파이프라인으로 실행해, 담보·직장조건 하드규칙이 안내문까지 정확히 반영되는지 확인합니다. (LLM 호출이 있어 시간이 걸립니다)

In [18]:
for tc in EDGE_CASES:
    print(f"\n\n########## {tc['name']} (기대: {tc['expected']}) ##########")
    out = await run_service(tc["input"])
    display(Markdown(f"#### [{tc['name']}] 최종 안내문\n\n" + out["안내문"]))



########## 담보 보유 저신용 케이스 (기대: 승인가능) ##########


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 94613b06-2629-4458-a7b7-1bab256ddde0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 다음 고객 입력을 파싱하라:                                                                               │
│  "월소득 300만원이고 부채는 500만원 있어요. 신용등급 8등급이고 800만원 대출받고 싶은데, 집을 담보로 제공할 수   │
│  있습니다."                                                                                                     │
│                                                                                                                 │
│  월소득·부채·신용등급·희망금액·직장유형·담보보유 6개 필드를 추출하라. 금액은 원 단위 정수로                     │
│  환산하라(700만원->7000000). 부채가 '없다'면 0. 직장유형은 정규직/계약직/제한없음 중 하나. 담보보유는 담보      │
│  제공 의사가 명시된 경우에만 true, 그 외 false.                                                                 │
│  ID: f0c7dd0f-a1f6-4f75-bc4c-b2d5a2fb417d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: 정보 파싱가                                                                                             │
│                                                                                                                 │
│  Task: 다음 고객 입력을 파싱하라:                                                                               │
│  "월소득 300만원이고 부채는 500만원 있어요. 신용등급 8등급이고 800만원 대출받고 싶은데, 집을 담보로 제공할 수   │
│  있습니다."                                                                                                     │
│                                                                                                                 │
│  월소득·부채·신용등급·희망금액·직장유형·담보보유 6개 필드를 추출하라. 금액은 원 단위 정수로                     │
│  환산하라(700만원->7000000). 부채가 '없다'면 0. 직장유형은 정규직/계약직/제한없음 중 하나. 담보보유는 담보      │
│  제공 의사가 명시된 경우에만 true, 그 외 false.                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

[Finalize] todos_count=0, todos_with_results=0


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Agent 2의 판정과 근거를 받아 고객용 안내문을 작성하라. Agent 2가 전달한 '추천상품'(도구가 최저금리       │
│  기준으로 확정한 시스템 추천)을 최우선으로 그대로 안내하라 — 스스로 다른 상품을 계산해 대체하거나 추가로 끼워   │
│  넣지 마라. 그 추천 상품 하나만 'lookup_loan_product' 도구로 조회해 금리·한도를 재확인하라(ReAct) — 불필요한    │
│  반복 조회로 토큰과 응답 시간을 낭비하지 않는다. 판정 라벨에 맞춰 톤과 첫 문장을 다르게 하라(라벨마다 하나만):  │
│  '승인가능'이면 긍정적 톤으로 시작하고 '상담이 필요하다'는 문장은 넣지 마라. '상담필요'이면 보완하면 승인       │
│  가능성이 있는 중립적 상태이니 '승인 가능성이 낮은 상황입니다' 같은 부정적 문장 대신 '추가로 확인·보완이        │
│  필요한 부분이 있어 상담을 안내드립니다'처럼 중립적으로 시작하라. '어려움'이면 '추천상품'이 없으므로(None)      │
│  상품을 나열하거나 '추천'하지 말고, '현재 기준으로는 승인이 어려운 것으로 판단됩니다(데모 기준)'처럼 시작한 뒤  │
│  상환능력·신용등급 개선 방향(부채 축소, 소득 안정화, 담보 제공, 소액부터 재신청 등)과 상담 채널 안내로          │
│  마무리하라. 확인된 수치만 사용하고, 확정 표현 대신 조건부 표현을 쓰라. 같은 이름의 상품이 여러 은행에 있을 수  │
│  있으니, 상품을 언급할 때 상품코드와 은행명을 함께 명시하라. 안내문 마지막 줄에 다음을 그대로 포함하라: "본     │
│  안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다."     │
│  Agent: 결과 안내가                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


------------------------------------------------------------
토큰 Usage (추론 파이프라인 비용 체감 — Self-Attention/추론 반복):
  total_tokens=191766 prompt_tokens=162558 cached_prompt_tokens=67584 completion_tokens=29208 reasoning_tokens=0 cache_creation_tokens=0 successful_requests=90
------------------------------------------------------------


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

#### [계약직 소액 케이스] 최종 안내문

검토 결과 승인 가능한 것으로 판단됩니다(데모 기준). 

추천 상품은 다음과 같습니다:
- **상품명**: 중금리대출
- **상품코드**: D-01
- **은행**: D은행
- **금리범위**: 8.0% ~ 15.0%
- **최대한도**: 20,000,000원

이 상품은 고객님의 신용등급과 소득 조건에 적합하며, 대출 진행이 가능할 것으로 보입니다. 

본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 94613b06-2629-4458-a7b7-1bab256ddde0                                                                       │
│  Final Output: 검토 결과 승인 가능한 것으로 판단됩니다(데모 기준).                                              │
│                                                                                                                 │
│  추천 상품은 다음과 같습니다:                                                                                   │
│  - **상품명**: 중금리대출                                                                                       │
│  - **상품코드**: D-01                                                                                           │
│  - **은행**: D은행                                                                                              │
│  - **금리범위**: 8.0% ~ 15.0%                                                                                   │
│  - **최대한도**: 20,000,000원                                                                                   │
│                                                                                                                 │
│  이 상품은 고객님의 신용등급과 소득 조건에 적합하며, 대출 진행이 가능할 것으로 보입니다.                        │
│                                                                                                                 │
│  본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

### 14-4. 실시간 입력 (ipywidgets)

하드코딩된 3종 케이스 외에, 직접 상담 문장을 입력해서 즉시 심사 결과를 확인할 수 있습니다. 버튼을 누르면 Agent 1 파싱 결과 → Agent 2 심사 결과(판정/CoT/SC/적격상품) → Agent 3 최종 안내문 → 토큰 Usage 순서로 출력됩니다.

In [19]:
import asyncio
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output

input_box = widgets.Textarea(
    placeholder="예) 월급 350만원 받는 정규직이고 부채는 800만원 있어요. 신용등급 3등급이고 2000만원 대출받고 싶어요.",
    description="고객 입력:",
    layout=widgets.Layout(width="100%", height="80px"),
    style={"description_width": "80px"},
)
run_button = widgets.Button(description="심사 실행", button_style="primary")
output_area = widgets.Output()


async def _run_and_render(text: str):
    with output_area:
        clear_output()
        print("실행 중... (Agent 1 파싱 → Agent 2 심사 → Agent 3 안내)")
    out = await run_service(text)
    with output_area:
        clear_output()
        print("[입력]", text, "\n")
        print("[Agent 1 파싱 결과]")
        print(out.get("파싱결과"), "\n")
        print("[Agent 2 심사 결과 — 판정/CoT/SC/적격상품]")
        print(out.get("심사결과"), "\n")
        display(Markdown("### Agent 3 최종 안내문\n\n" + out["안내문"]))
        print("\n[토큰 Usage]", out["usage"])


def _on_click(_button):
    text = input_box.value.strip()
    if not text:
        with output_area:
            clear_output()
            print("고객 입력을 먼저 작성해주세요.")
        return
    asyncio.create_task(_run_and_render(text))


run_button.on_click(_on_click)
display(input_box, run_button, output_area)

Textarea(value='', description='고객 입력:', layout=Layout(height='80px', width='100%'), placeholder='예) 월급 350만원 …

Button(button_style='primary', description='심사 실행', style=ButtonStyle())

Output()

---
## 15. 마무리 · 수업 개념 요약

| 개념 | 이 노트북에서의 위치 |
|---|---|
| **Self-Attention** | Agent 호출 시 프롬프트 전체 토큰 동시 참조 → 토큰 Usage(셀 14)로 체감 |
| **CoT** | Agent 2가 부채비율→신용등급→한도→상품선별 4단계 추론 |
| **Self-Consistency** | Agent 2가 보수·낙관·중립 3관점 교차검증(다회 호출 → 비용↑) |
| **ReAct** | Agent 3이 lookup_loan_product 조회 후 안내문 작성 |
| **추론 파이프라인 비용** | 3-Agent 순차 호출 = 토큰·지연 누적(usage_metrics 실측) |

> ⚠️ 본 안내는 교육용 데모이며 실제 대출 심사·법적·금융 자문이 아닙니다. 실제 대출은 각 금융기관 심사를 따릅니다.